In [2]:
import numpy as np
import os
import json
import time
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, accuracy_score
from joblib import dump

# Configuración de logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('pipeline_optimizer')

# Directorio para guardar modelos
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)


def load_data(models_dir=MODELS_DIR):
    """
    Carga los datos preprocesados de EEG
    """
    logger.info("Cargando datos preprocesados...")
    
    # Buscar archivos preprocesados
    x_files = [f for f in os.listdir(models_dir) if f.startswith('X_preprocessed') and f.endswith('.npy')]
    y_files = [f for f in os.listdir(models_dir) if f.startswith('y_labels') and f.endswith('.npy')]
    
    if not x_files or not y_files:
        raise FileNotFoundError(f"No se encontraron archivos de datos en {models_dir}")
    
    # Ordenar por timestamp para obtener los más recientes
    x_files.sort(reverse=True)
    y_files.sort(reverse=True)
    
    X = np.load(os.path.join(models_dir, x_files[0]))
    y = np.load(os.path.join(models_dir, y_files[0]))
    
    logger.info(f"Datos cargados: X shape {X.shape}, y shape {y.shape}")
    logger.info(f"Clases: {np.unique(y)}")
    
    # Intentar cargar info de preprocesamiento
    preprocessing_info = None
    info_files = [f for f in os.listdir(models_dir) if f.startswith('preprocessing_info') and f.endswith('.json')]
    if info_files:
        info_files.sort(reverse=True)
        with open(os.path.join(models_dir, info_files[0]), 'r') as f:
            preprocessing_info = json.load(f)
    
    return X, y, preprocessing_info


def create_pipeline_combinations():
    """
    Crea una selección balanceada de pipelines para clasificación de EEG
    """
    # Componentes para los pipelines
    scalers = {
        'standard': StandardScaler(),
        'robust': RobustScaler()
    }
    
    feature_processors = {
        'pca': PCA(n_components=0.95),
        'pca_fixed': PCA(n_components=15),
        'select_k': SelectKBest(f_classif, k=20)
    }
    
    classifiers = {
        'svc': SVC(probability=True, class_weight='balanced', random_state=42),
        'svc_linear': SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42),
        'rf': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
        'lr': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    }
    
    # Crear combinaciones de pipelines
    pipelines = {}
    
    for scaler_name, scaler in scalers.items():
        for feat_name, feat_processor in feature_processors.items():
            for clf_name, clf in classifiers.items():
                pipeline_name = f"{scaler_name}_{feat_name}_{clf_name}"
                
                pipeline = Pipeline([
                    ('scaler', scaler),
                    ('feature_processor', feat_processor),
                    ('classifier', clf)
                ])
                
                pipelines[pipeline_name] = pipeline
    
    # Añadir pipelines personalizados de alto rendimiento
    custom_pipelines = {
        'optimized_svc': Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95)),
            ('classifier', SVC(C=10, gamma='scale', kernel='rbf', probability=True, class_weight='balanced', random_state=42))
        ]),
        'optimized_rf': Pipeline([
            ('scaler', RobustScaler()),
            ('pca', PCA(n_components=20)),
            ('classifier', RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_split=2, class_weight='balanced', random_state=42))
        ])
    }
    
    pipelines.update(custom_pipelines)
    
    logger.info(f"Creados {len(pipelines)} pipelines para evaluación")
    return pipelines


def evaluate_pipelines(X, y, n_top=5):
    """
    Evalúa todos los pipelines y retorna los mejores
    """
    logger.info("Iniciando evaluación de pipelines...")
    
    # Crear pipelines
    pipelines = create_pipeline_combinations()
    
    # Configurar validación cruzada
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Métricas para evaluación
    scoring = {
        'accuracy': 'accuracy',
        'f1_weighted': make_scorer(f1_score, average='weighted')
    }
    
    # Evaluar cada pipeline
    results = []
    
    for name, pipeline in pipelines.items():
        logger.info(f"Evaluando pipeline: {name}")
        
        try:
            start_time = time.time()
            
            # Evaluación con validación cruzada
            scores = cross_val_score(
                pipeline, X, y,
                cv=cv,
                scoring='accuracy',
                n_jobs=-1  # Usar todos los cores disponibles
            )
            
            # Métricas adicionales
            f1_scores = cross_val_score(
                pipeline, X, y,
                cv=cv,
                scoring=make_scorer(f1_score, average='weighted'),
                n_jobs=-1
            )
            
            # Tiempo y resultados
            eval_time = time.time() - start_time
            mean_acc = np.mean(scores)
            std_acc = np.std(scores)
            mean_f1 = np.mean(f1_scores)
            
            logger.info(f"{name}: Accuracy = {mean_acc:.4f} ± {std_acc:.4f}, F1 = {mean_f1:.4f} (Tiempo: {eval_time:.2f}s)")
            
            # Guardar resultados
            results.append({
                'name': name,
                'accuracy': float(mean_acc),
                'acc_std': float(std_acc),
                'f1_score': float(mean_f1),
                'time': float(eval_time),
                'pipeline': pipeline
            })
            
        except Exception as e:
            logger.error(f"Error evaluando {name}: {str(e)}")
    
    # Ordenar por accuracy
    results.sort(key=lambda x: x['accuracy'], reverse=True)
    
    # Mostrar los mejores pipelines
    logger.info(f"\nMejores {n_top} pipelines:")
    for i, result in enumerate(results[:n_top]):
        logger.info(f"{i+1}. {result['name']}: Accuracy = {result['accuracy']:.4f}, F1 = {result['f1_score']:.4f}")
    
    return results


def train_best_pipeline(results, X, y, test_size=0.2):
    """
    Entrena el mejor pipeline con todos los datos y también
    genera métricas de test para compatibilidad con el script de predicción
    """
    if not results:
        logger.error("No hay resultados disponibles para entrenar el mejor pipeline")
        return None
    
    # Obtener el mejor pipeline
    best_result = results[0]
    best_pipeline_name = best_result['name']
    best_pipeline = best_result['pipeline']
    best_accuracy = best_result['accuracy']
    
    logger.info(f"\nEntrenando el mejor pipeline: {best_pipeline_name} (Accuracy: {best_accuracy:.4f})")
    
    # Dividir datos en train/test para una evaluación final
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    
    # Entrenar con datos de entrenamiento
    start_time = time.time()
    best_pipeline.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Evaluar en conjunto de test
    y_pred = best_pipeline.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average='weighted')
    test_precision = precision_score(y_test, y_pred, average='weighted')
    test_recall = recall_score(y_test, y_pred, average='weighted')
    
    logger.info(f"Métricas en test - Accuracy: {test_accuracy:.4f}, F1: {test_f1:.4f}")
    
    # Guardar datos de test para uso futuro (compatibilidad con script predict)
    np.save(os.path.join(MODELS_DIR, 'X_test.npy'), X_test)
    np.save(os.path.join(MODELS_DIR, 'y_test.npy'), y_test)
    np.save(os.path.join(MODELS_DIR, 'y_pred.npy'), y_pred)
    
    # Guardar el pipeline entrenado
    pipeline_path = os.path.join(MODELS_DIR, 'best_eeg_pipeline.joblib')
    dump(best_pipeline, pipeline_path)
    
    # También guardar como final_model_latest para compatibilidad con predict
    latest_path = os.path.join(MODELS_DIR, 'final_model_latest.joblib')
    dump(best_pipeline, latest_path)
    
    # Crear matriz de confusión y guardarla
    cm = confusion_matrix(y_test, y_pred)
    
    # Visualizar matriz de confusión
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de Confusión')
    plt.colorbar()
    
    classes = np.unique(y)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)
    
    # Añadir texto
    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    horizontalalignment="center",
                    color="white" if cm[i, j] > thresh else "black")
            
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Etiqueta Predicha')
    plt.tight_layout()
    
    # Crear directorio para resultados si no existe
    results_dir = os.path.join(MODELS_DIR, 'predictions')
    os.makedirs(results_dir, exist_ok=True)
    
    # Guardar matriz de confusión
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    plt.savefig(os.path.join(results_dir, f'confusion_matrix_{timestamp}.png'))
    
    # Guardar información
    pipeline_info = {
        'name': best_pipeline_name,
        'accuracy': float(best_accuracy),
        'test_accuracy': float(test_accuracy),
        'test_f1': float(test_f1),
        'test_precision': float(test_precision),
        'test_recall': float(test_recall),
        'training_time': float(training_time),
        'path': pipeline_path,
        'latest_path': latest_path,
        'date': time.strftime('%Y-%m-%d %H:%M:%S'),
        'confusion_matrix': cm.tolist()
    }
    
    # Guardar info de entrenamiento en formato compatible con predict
    training_info_path = os.path.join(MODELS_DIR, f'training_info_{timestamp}.json')
    training_info = {
        'timestamp': timestamp,
        'pipeline_type': 'Standard',
        'training_time': float(training_time),
        'test_metrics': {
            'accuracy': float(test_accuracy),
            'f1_score': float(test_f1),
            'precision': float(test_precision),
            'recall': float(test_recall)
        },
        'model_path': latest_path,
        'test_set_size': len(y_test),
        'train_set_size': len(y_train),
        'confusion_matrix': cm.tolist()
    }
    
    with open(training_info_path, 'w') as f:
        json.dump(training_info, f, indent=4)
    
    info_path = os.path.join(MODELS_DIR, 'pipeline_info.json')
    with open(info_path, 'w') as f:
        json.dump(pipeline_info, f, indent=4)
    
    logger.info(f"Pipeline guardado en: {pipeline_path}")
    logger.info(f"Pipeline también guardado como: {latest_path}")
    logger.info(f"Información de entrenamiento guardada en: {training_info_path}")
    
    return best_pipeline, pipeline_info


def visualize_results(results, title="Comparación de Pipelines"):
    """
    Visualiza los resultados de evaluación
    """
    if not results:
        logger.error("No hay resultados para visualizar")
        return
    
    # Tomar los 8 mejores resultados para visualización
    top_results = results[:8]
    
    # Datos para gráfico
    names = [r['name'] for r in top_results]
    accuracies = [r['accuracy'] for r in top_results]
    f1_scores = [r['f1_score'] for r in top_results]
    std_errors = [r['acc_std'] for r in top_results]
    
    # Invertir orden para mejor visualización
    names = names[::-1]
    accuracies = accuracies[::-1]
    f1_scores = f1_scores[::-1]
    std_errors = std_errors[::-1]
    
    # Crear gráfico
    fig, ax = plt.figure(figsize=(12, 8)), plt.subplot(111)
    
    # Barras para accuracy
    ax.barh(range(len(names)), accuracies, xerr=std_errors, 
            align='center', alpha=0.7, color='#2C82C9', label='Accuracy')
    
    # Línea para F1 score
    for i, (name, f1) in enumerate(zip(names, f1_scores)):
        ax.plot(f1, i, 'ro', markersize=8, label='F1 Score' if i == 0 else "")
    
    # Formato
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.set_xlabel('Puntuación')
    ax.set_title(title)
    ax.set_xlim(min(min(accuracies)-0.1, min(f1_scores)-0.1), 1.0)
    ax.grid(axis='x', linestyle='--', alpha=0.6)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR, 'pipeline_comparison.png'))
    logger.info(f"Gráfico guardado en: {os.path.join(MODELS_DIR, 'pipeline_comparison.png')}")


def main():
    """
    Función principal para encontrar el mejor pipeline de clasificación de EEG
    """
    import argparse
    
    # Añadir argumentos para ser compatible con el script de predicción
    parser = argparse.ArgumentParser(description='Optimizador de Pipelines para EEG')
    parser.add_argument('--mode', choices=['train', 'optimize', 'both'], default='both',
                       help='Modo: optimize solo evalúa pipelines, train entrena el mejor, both hace ambos')
    parser.add_argument('--test_size', type=float, default=0.2, help='Tamaño del conjunto de test')
    
    args = parser.parse_args()
    
    logger.info("=== Iniciando Optimizador de Pipelines para EEG ===")
    logger.info(f"Modo: {args.mode}, Test size: {args.test_size}")
    
    try:
        # Cargar datos preprocesados
        X, y, preprocessing_info = load_data()
        
        if args.mode in ['optimize', 'both']:
            # Evaluar pipelines
            results = evaluate_pipelines(X, y)
            
            # Visualizar resultados
            if results:
                visualize_results(results, "Comparación de Pipelines para Clasificación EEG")
            else:
                logger.error("No se encontraron pipelines válidos")
                return
        else:
            # Si estamos en modo 'train', usamos un pipeline predeterminado
            logger.info("Modo train: usando pipeline predeterminado (StandardScaler + PCA + SVC)")
            from sklearn.pipeline import Pipeline
            from sklearn.preprocessing import StandardScaler
            from sklearn.decomposition import PCA
            from sklearn.svm import SVC
            
            default_pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('feature_processor', PCA(n_components=0.95)),
                ('classifier', SVC(probability=True, class_weight='balanced', random_state=42))
            ])
            
            results = [{
                'name': 'standard_pca_svc',
                'accuracy': 0.0,  # No evaluado
                'acc_std': 0.0,
                'f1_score': 0.0,
                'time': 0.0,
                'pipeline': default_pipeline
            }]
        
        if args.mode in ['train', 'both'] and results:
            # Entrenar el mejor pipeline
            best_pipeline, pipeline_info = train_best_pipeline(results, X, y, test_size=args.test_size)
            
            # Guardar como best_pipeline.joblib para compatibilidad con el script predict
            compat_path = os.path.join(MODELS_DIR, 'best_pipeline.joblib')
            dump(best_pipeline, compat_path)
            
            # Guardar pipeline_info en formato compatible con predict
            compat_info = {
                'best_pipeline_name': pipeline_info['name'],
                'best_accuracy': pipeline_info['accuracy'],
                'training_time': pipeline_info['training_time'],
                'pipeline_path': compat_path,
                'is_csp': False,  # Por defecto no usamos CSP en este optimizador
                'evaluation_results': [{
                    'name': r['name'],
                    'accuracy': r.get('accuracy', 0.0),
                    'std': r.get('acc_std', 0.0),
                    'time': r.get('time', 0.0)
                } for r in results[:10]]
            }
            
            # Guardar info en formato compatible
            with open(os.path.join(MODELS_DIR, 'best_pipeline_info.json'), 'w') as f:
                json.dump(compat_info, f, indent=4)
            
            # Resumen
            logger.info("\n=== Resumen de Optimización ===")
            logger.info(f"Mejor pipeline: {pipeline_info['name']}")
            logger.info(f"Accuracy CV: {pipeline_info['accuracy']:.4f}")
            logger.info(f"Accuracy Test: {pipeline_info['test_accuracy']:.4f}")
            logger.info(f"F1 Test: {pipeline_info['test_f1']:.4f}")
            logger.info(f"Tiempo de entrenamiento: {pipeline_info['training_time']:.2f} segundos")
            logger.info(f"Modelo guardado en: {pipeline_info['path']}")
            logger.info(f"Modelo también guardado como: {pipeline_info['latest_path']}")
            logger.info("==============================")
            
    except Exception as e:
        logger.error(f"Error en el optimizador de pipelines: {str(e)}")
        raise


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--mode {train,optimize,both}]
                             [--test_size TEST_SIZE]
ipykernel_launcher.py: error: unrecognized arguments: --f=/home/codespace/.local/share/jupyter/runtime/kernel-v3e2e3faa2335b724f2bfbe97d98b12d798daf890c.json


SystemExit: 2

/workspaces/42_vortex/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3554: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
